> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# HS project 04 — mvBLUP model ladder

In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__

In [ ]:
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str);train.columns=train.columns.str.replace('regressedlr_','',regex=False)
test_rows=pd.read_csv(TEST_ROWS_CSV,index_col=0);test_rows.index=test_rows.index.astype(str);test_rows.columns=test_rows.columns.str.replace('regressedlr_','',regex=False);test_rows=test_rows.reindex(index=train.index,columns=train.columns)
test_elems=pd.read_csv(TEST_ELEMS_CSV,index_col=0);test_elems.index=test_elems.index.astype(str);test_elems.columns=test_elems.columns.str.replace('regressedlr_','',regex=False);test_elems=test_elems.reindex(index=train.index,columns=train.columns)
store=vigor.open_store(ANALYSIS_STORE,mode='a',lazy=False)
Ga=np.asarray(vigor.open_grm(store,component='additive'),float)
Gd=np.asarray(vigor.open_grm(store,component='dominance'),float)
fit_mask=train.notna().any(axis=1).to_numpy()

# Recreate the deterministic observed-environment kernel names used by notebook 02.
pdr=pd.read_csv(PROCESSED_DATA_CSV,dtype={'rfid':str}).set_index('rfid').reindex(train.index)
if GXE_ENVIRONMENT_COLUMN is not None:
    env=pdr[GXE_ENVIRONMENT_COLUMN]
else:
    sex=[c for c in pdr.columns if str(c).lower()=='sex']; env=pdr[sex[0]] if sex else None
if env is None: raise ValueError('Set GXE_ENVIRONMENT_COLUMN; no sex column was found.')
kernels,_=vigor.make_gxe_kernels({'additive':Ga,'dominance':Gd},env,index=train.index,fit_mask=fit_mask,include_gxe=True,gxe_components=GXE_COMPONENTS,dtype=np.float64)


In [ ]:
def _load_cov(group):
    ds=store.open_dataset(group,lazy=False)
    E=np.asarray(ds['E'],float)
    if 'component' in ds['G'].dims:
        G={str(c):np.asarray(ds['G'].sel(component=c),float) for c in ds.component.values}
    else:
        G=np.asarray(ds['G'],float)
    return G,E

def _predict(label,K,G,E):
    _mvb=vigor.mvBLUP(
        train,K,G,E=E,missing='exact',solver='pcg',solver_tol=MVBLUP_TOL,
        solver_maxiter=MVBLUP_MAXITER,preconditioner=MVBLUP_PRECONDITIONER,
        accuracy=MVBLUP_ACCURACY,kernel_rank='full',return_aux=True,blas_threads=BLAS_THREADS)
    pred = _mvb["total"].to_pandas(); aux = dict(_mvb.attrs)
    pred=pd.DataFrame(np.asarray(pred,float),index=train.index,columns=train.columns)
    store.write_predictions(pred,name='mvblup',group=f'prediction/{label}',mode='a')
    return pred,aux

models={}; diagnostics=[]
for label,group,K in [
    ('GE_rank10','covariance/GE_rank10',Ga),
    ('GE_full','covariance/GE_full',Ga),
    ('ADGxE_full','covariance/ADGxE_full',kernels),
]:
    G,E=_load_cov(group)
    pred,aux=_predict(label,K,G,E)
    models[label]=pred
    diagnostics.append({'model':label,'cg_iterations':aux.get('cg_iterations'),
                        'cg_relative_residual':aux.get('cg_relative_residual'),
                        'preconditioner':aux.get('preconditioner')})

display(pd.DataFrame(diagnostics))


In [ ]:
def trait_metrics(truth,pred,min_n=MIN_R2_N):
    rows=[]
    for trait in train.columns:
        y=truth[trait].to_numpy(float);p=pred[trait].to_numpy(float);m=np.isfinite(y)&np.isfinite(p)
        if not m.any(): continue
        rmse=float(np.sqrt(np.mean((y[m]-p[m])**2)))
        r2=np.nan
        if m.sum()>=min_n and np.std(y[m])>0 and np.std(p[m])>0:
            r2=float(np.corrcoef(y[m],p[m])[0,1]**2)
        rows.append({'trait':trait,'n':int(m.sum()),'r2':r2,'rmse':rmse})
    return pd.DataFrame(rows)

rows=[]; bytrait=[]
for split,truth in {'Fully unseen animals':test_rows,'Seen animals, unseen traits':test_elems}.items():
    for name,pred in models.items():
        z=trait_metrics(truth,pred); z['split']=split; z['model']=name; bytrait.append(z)
        rows.append({'split':split,'model':name,'median_r2':z.r2.median(),'mean_r2':z.r2.mean(),
                     'median_rmse':z.rmse.median(),'n_traits':int(z.r2.notna().sum())})
summary=pd.DataFrame(rows)
summary.to_csv(RESULTS/'model_ladder_summary.csv',index=False)
pd.concat(bytrait,ignore_index=True).to_csv(RESULTS/'model_ladder_by_trait.csv',index=False)
pd.DataFrame(diagnostics).to_csv(RESULTS/'mvblup_pcg_diagnostics.csv',index=False)
display(summary)


This notebook is the accuracy reference. Add low-rank L-BFGS, A+D+G×E, rrBLUP and the historical gdec comparator as rows, but keep the **same phenotype mask, kernels and solver tolerance**.